In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "mistralai/Mistral-7B-v0.1"
adapter_id = "caffeic/text-to-sql-model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_id, subfolder="checkpoint-484")

tokenizer = AutoTokenizer.from_pretrained(adapter_id)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

checkpoint-484/adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/463 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def test_sql_generation(question):
    prompt = f"### Instruction:\nConvert natural language to SQL\n\n### Input:\n{question}\n\n### Output:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql_result = full_output.split("### Output:\n")[-1].strip()
    return sql_result

In [ ]:
example_question = "List the head of each department and the number of employees working there."
print(f"Question: {example_question}")
print(f"Generated SQL: {test_sql_generation(example_question)}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Question: List the head of each department and the number of employees working there.
Generated SQL: SELECT T1.dept_name ,  COUNT(*) FROM department AS T1 JOIN employee AS T2 ON T1.dept_id  =  T2.dept_id GROUP BY T1.dept_name


In [ ]:
from datasets import load_dataset
ds = load_dataset("gretelai/synthetic_text_to_sql")

README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 5851
    })
})

In [ ]:
ds['test'][0]

{'id': 1,
 'domain': 'artificial intelligence',
 'domain_description': 'AI data on algorithmic fairness, AI safety, explainable AI, and creative AI applications.',
 'sql_complexity': 'basic SQL',
 'sql_complexity_description': 'basic SQL with a simple select statement',
 'sql_task_type': 'analytics and reporting',
 'sql_task_type_description': 'generating reports, dashboards, and analytical insights',
 'sql_prompt': "What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?",
 'sql_context': "CREATE TABLE creative_ai (application_id INT, name TEXT, region TEXT, explainability_score FLOAT); INSERT INTO creative_ai (application_id, name, region, explainability_score) VALUES (1, 'ApplicationX', 'Europe', 0.87), (2, 'ApplicationY', 'North America', 0.91), (3, 'ApplicationZ', 'Europe', 0.84), (4, 'ApplicationAA', 'North America', 0.93), (5, 'ApplicationAB', 'Europe', 0.89);",
 'sql': "SELECT AVG(explainability_score) FRO

In [ ]:
example = ds['train'][0]
print(f"PROMPT: {example['sql_prompt']}")
print(f"CONTEXT: {example['sql_context']}")
print(f"EXPLANATION: {example['sql_explanation']}")

PROMPT: What is the total volume of timber sold by each salesperson, sorted by salesperson?
CONTEXT: CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
EXPLANATION: Joins timber_sales and salesperson tables, groups sales by salesperson, calculates total volume sold by each salesperson, and orders the results by total volume in descending order.


In [ ]:
import pandas as pd

In [ ]:
new_df = pd.DataFrame(ds['test'].select(range(100))) # look at first 100
print(new_df['sql_complexity'].value_counts())

sql_complexity
basic SQL           47
aggregation         26
single join         11
subqueries           9
window functions     6
multiple_joins       1
Name: count, dtype: int64


In [ ]:
from tqdm import tqdm

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm

def build_prompt(question, context=None):
    if context:
        return f"""### Instruction: Convert natural language to SQL ### Context: {context} ### Input: {question} ### Output: """
    else:
        return f"""### Instruction: Convert natural language to SQL ### Input: {question} ## Output: """

def extract_sql(text):
    if "### Output:" in text:
        return text.split("### Output:")[-1].strip()
    return text.strip()

def generate_sql(prompt, model, tokenizer, device="cuda"):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,  # deterministic
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_sql(decoded)

In [ ]:
# =========================
# 4. METRICS
# =========================
def exact_match(pred, gt):
    return pred.strip().lower() == gt.strip().lower()


def execution_match(pred_sql, gt_sql, conn):
    try:
        pred_res = conn.execute(pred_sql).fetchall()
        gt_res = conn.execute(gt_sql).fetchall()
        return pred_res == gt_res
    except:
        return False

In [ ]:
# =========================
# 5. MAIN EVALUATION LOOP
# =========================
def evaluate(ds, model, tokenizer, num_samples=None, use_context=True, db_conn=None, device="cuda", save_path="results.csv" ):
    results = []

    test_data = ds["test"]
    total = len(test_data) if num_samples is None else num_samples

    exact_matches = 0
    exec_matches = 0

    for i in tqdm(range(total)):
        sample = test_data[i]

        question = sample["sql_prompt"]
        ground_truth = sample["sql"]
        context = sample.get("sql_context", None)

        prompt = build_prompt(question, context if use_context else None)
        prediction = generate_sql(prompt, model, tokenizer, device)

        em = exact_match(prediction, ground_truth)
        exact_matches += em

        if db_conn:
            ex = execution_match(prediction, ground_truth, db_conn)
            exec_matches += ex
        else:
            ex = None

        results.append({
            "id": sample["id"],
            "question": question,
            "context": context,
            "predicted_sql": prediction,
            "ground_truth_sql": ground_truth,
            "exact_match": em,
            "execution_match": ex
        })

    # =========================
    # METRICS SUMMARY
    # =========================
    exact_acc = exact_matches / total

    print("\n===== RESULTS =====")
    print(f"Total Samples: {total}")
    print(f"Exact Match Accuracy: {exact_acc:.4f}")

    if db_conn:
        exec_acc = exec_matches / total
        print(f"Execution Accuracy: {exec_acc:.4f}")

    # =========================
    # SAVE CSV
    # =========================
    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    print(f"\nSaved results to {save_path}")

    return df

In [ ]:
def evaluate_metrics(ds, model, tokenizer, n=1000, device="cuda"):
    correct = 0

    for i in range(n):
        sample = ds["test"][i]

        question = sample["sql_prompt"]
        gt = sample["sql"]

        prompt = f"""### Instruction:
Convert natural language to SQL

### Input:
{question}

### Output:
"""

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = pred.split("### Output:")[-1].strip()

        if pred.strip().lower() == gt.strip().lower():
            correct += 1

    acc = correct / n
    print(f"Exact Match Accuracy: {acc:.4f}")
    return acc

In [ ]:
evaluate_metrics(ds, model, tokenizer, n=500)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o